# Checked: Contracting

This page runs the checks that the Checked block of the
[Contracting](../docs/contracting.md) chapter claims: eight S-shapes over the
measles record, three M-shapes over the model graph, and the counterexamples
that must fail. A reader need not take the chapter's word for it. Every line
of output below is produced by the code above it, the gate re-executes this
notebook and fails if the committed outputs differ, and each claim is an
`assert` that stops the notebook when it does not hold.

In [1]:
import sys; sys.path[:0] = ["notebooks", "."]  # the shared module lives beside this notebook
import checked

# The shapes the chapter's Checked block names, by file.
RECORD_SHAPES = ["S0-Parties", "S0-Access", "S0-Population", "S0-Need", "S0-StatementOfWork", "S0-Proposal", "S0-Layers", "S9-Acceptance"]
MODEL_SHAPES = ["M1-Parties", "M1-Obligation", "M5-Cardinality"]

## The record conforms

The record is `track/measles-run.ttl` with the EPO vocabulary, loaded as the
test suite loads it. The shapes graph holds exactly the eight named shapes
copied from `shapes/epo.shapes.ttl`; a name that is not a shape in that file
raises an error, so a misnamed claim cannot pass silently.

In [2]:
record = checked.record()
S = checked.shapes("shapes/epo.shapes.ttl", RECORD_SHAPES)
conforms, fired = checked.report("track/measles-run.ttl", record, S)
assert conforms and not fired
checked.passed("the measles record conforms to the eight S-shapes the chapter names")

track/measles-run.ttl: conforms = True
  S0-Access            pass
  S0-Layers            pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        pass
  S0-Proposal          pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
ok: the measles record conforms to the eight S-shapes the chapter names


## A requirement set before the agreement

The chapter says a requirement set declared before the agreement was signed
fails the layer rule, `S0-Layers`. The counterexample is
`counterexamples/requirements-before-agreement.ttl`, run against the same
eight shapes. `S0-Parties` fires as well: it carries its own ordering
constraint, that the agreement precedes the requirement set (R-21), and the
layer rule (R-32) generalises it to every pinned item.

In [3]:
cx = checked.counterexample("requirements-before-agreement.ttl")
conforms, fired = checked.report("counterexamples/requirements-before-agreement.ttl", cx, S)
assert not conforms and "S0-Layers" in fired
checked.passed("requirements-before-agreement.ttl does not conform and fails S0-Layers")

counterexamples/requirements-before-agreement.ttl: conforms = False
  S0-Access            pass
  S0-Layers            FAIL
  S0-Need              pass
  S0-Parties           FAIL
  S0-Population        pass
  S0-Proposal          pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Layers at cx:agreement:
    S0 two layers (R-32): every item pinned at the contract is generated no later than the requirement set, and every item pinned within the evaluation no earlier than the agreement.
  S0-Parties at cx:agreement:
    S0: the agreement precedes the requirement set: requirements are agreed under the contract, not before it (R-21).
ok: requirements-before-agreement.ttl does not conform and fails S0-Layers


## A population neither interviewed nor represented

`counterexamples/population-unrepresented.ttl` names an affected population
with no stakeholder input attributed to it and no domain expert representing
it. It must fail `S0-Population` and nothing else among the eight.

In [4]:
cx = checked.counterexample("population-unrepresented.ttl")
conforms, fired = checked.report("counterexamples/population-unrepresented.ttl", cx, S)
assert not conforms and set(fired) == {"S0-Population"}
checked.passed("population-unrepresented.ttl does not conform and fails S0-Population only")

counterexamples/population-unrepresented.ttl: conforms = False
  S0-Access            pass
  S0-Layers            pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        FAIL
  S0-Proposal          pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Population at cx:forgotten:
    S0: an affected population is engaged as the statement of work decided (R-21, R-40): interviewed, with a stakeholder input attributed to it, or represented by a named domain expert; and there is a decision for it.
ok: population-unrepresented.ttl does not conform and fails S0-Population only


A population the statement of work said would be interviewed, but the record only represents (ruling R-40): the decision was made, the evaluation did not realize it.

In [5]:
cx = checked.counterexample("engagement-mismatch.ttl")
conforms, fired = checked.report("counterexamples/engagement-mismatch.ttl", cx, S)
assert not conforms and set(fired) == {"S0-Population"}
checked.passed("engagement-mismatch.ttl does not conform and fails S0-Population only")

counterexamples/engagement-mismatch.ttl: conforms = False
  S0-Access            pass
  S0-Layers            pass
  S0-Need              pass
  S0-Parties           pass
  S0-Population        FAIL
  S0-Proposal          pass
  S0-StatementOfWork   pass
  S9-Acceptance        pass
  S0-Population at cx:population:
    S0: an affected population is engaged as the statement of work decided (R-21, R-40): interviewed, with a stakeholder input attributed to it, or represented by a named domain expert; and there is a decision for it.
ok: engagement-mismatch.ttl does not conform and fails S0-Population only


## The model graph conforms

The model graph is `model/og-caie.model.ttl`, the pruned RDF rendering of the
SysML source (R-22). The three named M-shapes are copied from
`shapes/model.shapes.ttl`.

In [6]:
model = checked.model_graph()
M = checked.shapes("shapes/model.shapes.ttl", MODEL_SHAPES)
conforms, fired = checked.report("model/og-caie.model.ttl", model, M)
assert conforms and not fired
checked.passed("the model graph conforms to the three M-shapes the chapter names")

model/og-caie.model.ttl: conforms = True
  M1-Obligation        pass
  M1-Parties           pass
  M5-Cardinality       pass
ok: the model graph conforms to the three M-shapes the chapter names


## A model with no obligation

`counterexamples/model/no-obligation.sysml` names a sponsor and affected
populations but relates them by nothing, and its mission regards no
population. It is built to RDF through the same pipeline as the canonical
graph (the pinned OpenSysML converter, then `scripts/prune_model.py`), and
must fail `M1-Obligation`. The package is minimal, so its testing
organization declares no account executive and `M5-Cardinality` fires too;
the claim is about `M1-Obligation`, and the assert says exactly that.

In [7]:
cx = checked.model_counterexample("no-obligation.sysml")
conforms, fired = checked.report("counterexamples/model/no-obligation.sysml", cx, M)
assert not conforms and "M1-Obligation" in fired
checked.passed("no-obligation.sysml does not conform and fails M1-Obligation")

counterexamples/model/no-obligation.sysml: conforms = False
  M1-Obligation        FAIL
  M1-Parties           pass
  M5-Cardinality       FAIL
  M1-Obligation at elmt:NoObligation__Mission:
    The mission regards one or more affected populations: the Mission item owns a reference part typed AffectedPopulation with lower bound one (R-37, R-38).
  M1-Obligation at elmt:NoObligation__OgCaieEvaluation:
    The assembly relates the sponsor to the affected populations by an obligation: a connection usage typed Obligation whose ends are the sponsor part and the affected part, a relation that carries no item (R-38).
  M5-Cardinality at elmt:NoObligation__TestingOrganization:
    The testing organization holds exactly one account executive (R-23).
ok: no-obligation.sysml does not conform and fails M1-Obligation


## Verdict

One line for the reader and for the gate. It is printed only when every cell
above ran and every assert held.

In [8]:
checked.verdict()

claims checked: 6
  the measles record conforms to the eight S-shapes the chapter names
  requirements-before-agreement.ttl does not conform and fails S0-Layers
  population-unrepresented.ttl does not conform and fails S0-Population only
  engagement-mismatch.ttl does not conform and fails S0-Population only
  the model graph conforms to the three M-shapes the chapter names
  no-obligation.sysml does not conform and fails M1-Obligation
NOTEBOOK: PASS
